# Encoder-decoder (simplified U-Net)

Part of **FL-YOLO-DADP** (Chen et al., *Improved Federated Learning with Differential Privacy for YOLO Detector on Phase Images*, CDSR 2025). This is one of the four generative baselines in Table 1.

The generator learns to map a 3-channel bounding-box mask (one channel per class: RBC, WBC, PLT) to a phase image. It is trained on real phase images with YOLO labels (`generator_train`), then used to generate a synthetic version of every frame of the three agents' train/valid splits.

**Cells:** *Train* → *Synthesize* (visual check on one frame) → *Generate* (writes `../data/synthetic_raw/encoder-decoder/agent_<k>_synthesized/`). Afterwards, from the repository root:

```bash
python data_prep/build_synthetic_dataset.py --raw data/synthetic_raw/encoder-decoder --agents data/agents --out data/synthetic/encoder-decoder
python baselines/train_yolo.py --data data/synthetic/encoder-decoder/data.yaml --epochs 20 --name synthetic_encoder-decoder
```

Paths are relative to this folder (`generative_models/`):

| path | content |
|---|---|
| `../data/generator_train/{train,valid,test}` | real frames + YOLO labels used to train the generator |
| `../data/agents/agent_<k>` | per-agent shards (input of *Generate*) |
| `../data/synthetic_raw/encoder-decoder` | output of *Generate* |
| `../runs/generative_models/encoder-decoder` | generator checkpoints |


In [ ]:
import os
os.makedirs("../runs/generative_models/encoder-decoder", exist_ok=True)


### Train

In [ ]:
import os
import glob
import cv2
import numpy as np
from PIL import Image
from tqdm import tqdm
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Set random seed for reproducibility
seed = 99
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ---------------------------
# 1. Dataset
# ---------------------------
class YOLOConditionalDataset(Dataset):
    """
    Loads images & YOLO-format bounding-box annotations,
    creates a multi-channel mask (one channel per class).
    """

    def __init__(self, 
                 images_dir, 
                 labels_dir, 
                 image_size=(512, 384),
                 num_classes=3,
                 transforms=None):
        super().__init__()
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.image_size = image_size  # (width=512, height=384)
        self.num_classes = num_classes
        self.transforms = transforms
        
        # Collect all image paths
        self.image_paths = sorted(glob.glob(os.path.join(self.images_dir, "*.png")))
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # 1. Load image
        img_path = self.image_paths[idx]
        filename = os.path.splitext(os.path.basename(img_path))[0]
        
        # 2. Construct corresponding label path
        label_path = os.path.join(self.labels_dir, filename + ".txt")
        
        # 3. Read image
        image = Image.open(img_path).convert("RGB")
        
        # 4. Create bounding box mask
        mask = self.create_mask(label_path, image.size)
        
        # 5. Resize image & mask to desired dimension
        image = image.resize(self.image_size, Image.BILINEAR)
        mask = cv2.resize(mask, self.image_size, interpolation=cv2.INTER_NEAREST)
        
        # 6. Convert PIL image & mask to tensors
        if self.transforms is not None:
            image = self.transforms(image)
        else:
            # Basic transform: to tensor, scale to [0,1]
            image = T.ToTensor()(image)
        
        # mask shape: (H, W, num_classes), convert to shape: (num_classes, H, W)
        # then convert to torch tensor
        mask = np.transpose(mask, (2, 0, 1))  # channels first
        mask = torch.from_numpy(mask).float()
        
        return mask, image
    
    def create_mask(self, label_path, original_size):
        """
        Create a multi-channel mask (H, W, num_classes).
        original_size: (width, height) of the image.
        
        YOLO format: class_id x_center y_center width height
        (normalized from 0 to 1).
        """
        w, h = original_size
        mask = np.zeros((h, w, self.num_classes), dtype=np.uint8)
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            for line in lines:
                class_id, x_c, y_c, bw, bh = line.strip().split()
                class_id = int(class_id)
                x_c, y_c, bw, bh = float(x_c), float(y_c), float(bw), float(bh)
                
                # Convert normalized coords back to pixel coords
                x_center = int(x_c * w)
                y_center = int(y_c * h)
                box_w = int(bw * w)
                box_h = int(bh * h)
                
                # YOLO bounding box coordinates
                x1 = max(0, x_center - box_w // 2)
                y1 = max(0, y_center - box_h // 2)
                x2 = min(w, x_center + box_w // 2)
                y2 = min(h, y_center + box_h // 2)
                
                # Draw bounding box area in the corresponding channel
                mask[y1:y2, x1:x2, class_id] = 1
        
        return mask

# ---------------------------
# 2. Encoder-Decoder Model (U-Net-like)
# ---------------------------
class DoubleConv(nn.Module):
    """(Conv => ReLU => Conv => ReLU)"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        return self.conv(x)

class UNetEncoderDecoder(nn.Module):
    """
    A simplified U-Net style model for conditional generation:
      - Input: 3-channel bounding box mask
      - Output: 3-channel image (RGB)
    """
    def __init__(self, in_channels=3, out_channels=3, features=[64, 128, 256, 512]):
        super(UNetEncoderDecoder, self).__init__()
        
        # Downsampling path
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature
        
        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        
        # Upsampling path
        rev_features = features[::-1]
        for feature in rev_features:
            self.ups.append(
                nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2)
            )
            self.ups.append(DoubleConv(feature*2, feature))
        
        # Final output layer
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
        
        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
    def forward(self, x):
        # Encoder
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Decoder
        skip_connections = skip_connections[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip_connection = skip_connections[i//2]
            
            # Concatenate skip connection
            x = torch.cat((skip_connection, x), dim=1)
            x = self.ups[i+1](x)
        
        return self.final_conv(x)

# ---------------------------
# 3. Training
# ---------------------------
def train_model():
    # Paths (adjust these!)
    train_images_dir = "../data/generator_train/train/images"
    train_labels_dir = "../data/generator_train/train/labels"
    val_images_dir = "../data/generator_train/valid/images"
    val_labels_dir = "../data/generator_train/valid/labels"
    
    # Hyperparameters
    batch_size = 4
    lr = 1e-4
    num_epochs = 20
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Dataset & DataLoader
    train_transforms = T.Compose([
        T.ToTensor()
    ])
    
    train_dataset = YOLOConditionalDataset(
        images_dir=train_images_dir,
        labels_dir=train_labels_dir,
        image_size=(512, 384),
        transforms=train_transforms
    )
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    
    val_dataset = YOLOConditionalDataset(
        images_dir=val_images_dir,
        labels_dir=val_labels_dir,
        image_size=(512, 384),
        transforms=train_transforms
    )
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    # Model
    model = UNetEncoderDecoder(in_channels=3, out_channels=3).to(device)
    
    # Loss & Optimizer
    criterion = nn.L1Loss()  # or nn.MSELoss(), or combination with adversarial loss in future
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Track best validation accuracy
    best_val_acc = -1.0
    
    # Training Loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        running_accuracy = 0.0
        
        # Wrap the train_loader with tqdm
        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for i, (mask, real_img) in enumerate(train_bar):
            mask, real_img = mask.to(device), real_img.to(device)
            
            optimizer.zero_grad()
            fake_img = model(mask)
            loss = criterion(fake_img, real_img)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            # Calculate accuracy (if applicable)
            # For example, you can use PSNR or SSIM for image quality metrics
            # Here, we just use L1 loss as a placeholder for accuracy
            running_accuracy += 1 - loss.item()  # Placeholder for accuracy
            
            # Update progress bar
            train_bar.set_postfix({
                'Train Loss': running_loss / (i + 1),
                'Train Acc': running_accuracy / (i + 1)
            })
        
        avg_train_loss = running_loss / len(train_loader)
        avg_train_acc = running_accuracy / len(train_loader)
        
        # Validation Loop
        model.eval()
        val_loss = 0.0
        val_accuracy = 0.0
        
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
            for i, (mask, real_img) in enumerate(val_bar):
                mask, real_img = mask.to(device), real_img.to(device)
                
                fake_img = model(mask)
                loss = criterion(fake_img, real_img)
                
                val_loss += loss.item()
                val_accuracy += 1 - loss.item()  # Placeholder for accuracy
                
                # Update progress bar
                val_bar.set_postfix({
                    'Val Loss': val_loss / (i + 1),
                    'Val Acc': val_accuracy / (i + 1)
                })
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_acc = val_accuracy / len(val_loader)
        
        print(f"Epoch [{epoch+1}/{num_epochs}], "
              f"Train Loss: {avg_train_loss:.4f}, Train Acc: {avg_train_acc:.4f}, "
              f"Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.4f}")
        
        # Save the last epoch weights
        last_save_path = "../runs/generative_models/encoder-decoder/encoder_decoder_size512-384_feature512_last.pth"
        torch.save(model.state_dict(), last_save_path)
        
        # Save the best epoch weights
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            best_save_path = "../runs/generative_models/encoder-decoder/encoder_decoder_size512-384_feature512_best.pth"
            torch.save(model.state_dict(), best_save_path)
            print(f"Best model saved to: {best_save_path} with Val Acc: {best_val_acc:.4f}")
    
    print(f"Training complete. Last model saved to: {last_save_path}")

if __name__ == "__main__":
    train_model()

### Synthesize

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

test_image = cv2.imread('../data/generator_train/test/images/test-agg1-2067.png')
test_image_np = np.array(test_image)
print(test_image_np.shape)
# Convert from BGR to RGB
test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
# Convert to float and scale to [0,1]
test_image = test_image.astype(np.float32) / 255.0
plt.axis("off")
plt.imshow(test_image)

In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
import matplotlib.pyplot as plt

device = "cuda"  # or "cpu"

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        return self.conv(x)

class UNetEncoderDecoder(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, features=[64, 128, 256, 512]):
        super(UNetEncoderDecoder, self).__init__()
        
        # Downsampling path
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        
        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature
        
        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        
        # Upsampling path
        rev_features = features[::-1]
        for feature in rev_features:
            self.ups.append(
                nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2)
            )
            self.ups.append(DoubleConv(feature*2, feature))
        
        # Final output layer
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
        
        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
    def forward(self, x):
        # Encoder
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)
        
        # Bottleneck
        x = self.bottleneck(x)
        
        # Decoder
        skip_connections = skip_connections[::-1]
        for i in range(0, len(self.ups), 2):
            x = self.ups[i](x)
            skip_connection = skip_connections[i//2]
            
            # Concatenate skip connection
            x = torch.cat((skip_connection, x), dim=1)
            x = self.ups[i+1](x)
        
        return self.final_conv(x)

# Load model (ensure UNetEncoderDecoder is defined or imported)
model = UNetEncoderDecoder(in_channels=3, out_channels=3).to(device)
model.load_state_dict(torch.load("../runs/generative_models/encoder-decoder/encoder_decoder_size512-384_feature512_best.pth", map_location=device))
model.eval()

# Read a test image (currently used as if it were the 3-channel mask)
test_image = cv2.imread('../data/generator_train/test/images/test-agg1-2067.png')
# Convert from BGR to RGB
test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
# Convert to float and scale to [0,1]
test_image = test_image.astype(np.float32) / 255.0

# [H, W, C] --> [C, H, W], then add batch dimension
mask_tensor = torch.from_numpy(np.transpose(test_image, (2, 0, 1))).unsqueeze(0).to(device)

# Generate synthetic image
with torch.no_grad():
    fake_image = model(mask_tensor)  # shape: [1, 3, H, W]

# Convert to NumPy for display
fake_image_np = fake_image.squeeze(0).cpu().numpy()  # shape: [3, H, W]
fake_image_np = np.transpose(fake_image_np, (1, 2, 0))  # shape: [H, W, 3]
fake_image_np = np.clip(fake_image_np, 0, 1)
gray = fake_image_np.mean(axis=2, keepdims=True)  # shape (H,W,1)
fake_image_np = np.repeat(gray, 3, axis=2)   # shape (H,W,3)
print('out_np.shape', fake_image_np.shape)  # Should be [H, W, 3]
print('out_np min max values', np.min(fake_image_np), np.max(fake_image_np))

identical_across_channels = (fake_image_np[:,:,0] == fake_image_np[:,:,1]).all() \
                            and (fake_image_np[:,:,1] == fake_image_np[:,:,2]).all()
if identical_across_channels:
    print("Now all channels are identical across each pixel.")
else:
    print("Something went wrong; channels differ.")

plt.imshow(fake_image_np)
plt.axis("off")
plt.show()

### Generate

In [ ]:
import os
import shutil
import torch
import torch.nn as nn
import numpy as np
import cv2

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

########################################
# 1. Define the Model Architecture
########################################
class DoubleConv(nn.Module):
    """(Conv => ReLU => Conv => ReLU)"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        
    def forward(self, x):
        return self.conv(x)

class UNetEncoderDecoder(nn.Module):
    """
    A simplified U-Net style model for conditional generation:
      - Input: 3-channel bounding box mask
      - Output: 3-channel image (RGB)
    """
    def __init__(self, in_channels=3, out_channels=3, features=[64, 128, 256, 512]):
        super(UNetEncoderDecoder, self).__init__()
        
        # Downsampling path
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        
        in_ch = in_channels
        for feature in features:
            self.downs.append(DoubleConv(in_ch, feature))
            in_ch = feature
        
        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        
        # Upsampling path
        rev_features = features[::-1]
        for feature in rev_features:
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature*2, feature))
        
        # Final output layer
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
        
        # Pooling
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
    def forward(self, x):
        # Encoder
        skip_connections = []
        current = x
        for down in self.downs:
            current = down(current)
            skip_connections.append(current)
            current = self.pool(current)
        
        # Bottleneck
        current = self.bottleneck(current)
        
        # Decoder
        skip_connections = skip_connections[::-1]
        for i in range(0, len(self.ups), 2):
            current = self.ups[i](current)
            skip_connection = skip_connections[i // 2]
            
            # Concatenate skip connection
            current = torch.cat((skip_connection, current), dim=1)
            current = self.ups[i + 1](current)
        
        return self.final_conv(current)

########################################
# 2. Load the Trained Model
########################################
def load_trained_model(model_path, device):
    model = UNetEncoderDecoder(in_channels=3, out_channels=3).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

########################################
# 3. Synthesize Images (with grayscale post-processing)
########################################
def synthesize_images(model, source_images_dir, target_images_dir, image_size=(512, 384)):
    """
    Synthesize images from bounding-box masks. Then force the final image
    to have identical values in each of the 3 channels (true grayscale).
    """
    os.makedirs(target_images_dir, exist_ok=True)
    
    # Collect .png images
    image_paths = sorted(
        [os.path.join(source_images_dir, fname) 
         for fname in os.listdir(source_images_dir)
         if fname.endswith('.png')]
    )
    
    for img_path in image_paths:
        # 1) Load & Preprocess Input (treated as bounding-box mask)
        test_image = cv2.imread(img_path)
        test_image = cv2.cvtColor(test_image, cv2.COLOR_BGR2RGB)
        test_image = test_image.astype(np.float32) / 255.0  # [0,1]
        
        # Resize
        test_image = cv2.resize(test_image, image_size, interpolation=cv2.INTER_LINEAR)
        
        # Convert to tensor: [H,W,C] -> [C,H,W], then batch dimension
        mask_tensor = torch.from_numpy(np.transpose(test_image, (2, 0, 1))).unsqueeze(0).to(device)
        
        # 2) Forward pass to synthesize
        with torch.no_grad():
            fake_image = model(mask_tensor)  # shape: [1, 3, H, W]
        
        # 3) Convert back to NumPy
        fake_image_np = fake_image.squeeze(0).cpu().numpy()  # [3,H,W]
        fake_image_np = np.transpose(fake_image_np, (1, 2, 0))  # [H,W,3]
        fake_image_np = np.clip(fake_image_np, 0, 1)
        
        # 4) Force grayscale by averaging channels
        gray = np.mean(fake_image_np, axis=2, keepdims=True)    # shape [H,W,1]
        fake_image_np = np.repeat(gray, 3, axis=2)              # shape [H,W,3]
        
        # 5) Scale to [0,255] and save
        fake_image_np = (fake_image_np * 255).astype(np.uint8)
        
        out_name = os.path.basename(img_path)
        out_path = os.path.join(target_images_dir, out_name)
        cv2.imwrite(out_path, cv2.cvtColor(fake_image_np, cv2.COLOR_RGB2BGR))

########################################
# 4. Copy Images and Labels
########################################
def copy_images_and_labels(source_dir, target_dir):
    os.makedirs(target_dir, exist_ok=True)
    
    # Copy images
    for fname in os.listdir(source_dir):
        if fname.endswith('.png'):
            src = os.path.join(source_dir, fname)
            dst = os.path.join(target_dir, fname)
            shutil.copy(src, dst)
    
    # Copy labels
    labels_source_dir = source_dir.replace('images', 'labels')
    labels_target_dir = target_dir.replace('images', 'labels')
    os.makedirs(labels_target_dir, exist_ok=True)
    
    for fname in os.listdir(labels_source_dir):
        if fname.endswith('.txt'):
            src_lbl = os.path.join(labels_source_dir, fname)
            if os.path.exists(src_lbl):
                dst_lbl = os.path.join(labels_target_dir, fname)
                shutil.copy(src_lbl, dst_lbl)

########################################
# 5. Main Function to Generate Dataset
########################################
def generate_synthesized_dataset():
    base_source_dir = "../data/agents"
    base_target_dir = "../data/synthetic_raw/encoder-decoder"
    
    agents = ['agent_0', 'agent_1', 'agent_2']
    
    # Load the trained model
    model_path = "../runs/generative_models/encoder-decoder/encoder_decoder_size512-384_feature512_best.pth"
    model = load_trained_model(model_path, device)
    
    # For each agent, synthesize that agent's images and copy the rest.
    for agent in agents:
        # Train data
        src_images_train = os.path.join(base_source_dir, agent, 'train', 'images')
        tgt_images_train = os.path.join(base_target_dir, f"{agent}_synthesized", 'train', 'images')
        synthesize_images(model, src_images_train, tgt_images_train)
        
        # Copy images+labels from other agents (train)
        for other_agent in agents:
            if other_agent != agent:
                other_src_train = os.path.join(base_source_dir, other_agent, 'train', 'images')
                copy_images_and_labels(other_src_train, tgt_images_train)
        
        # Copy labels from all agents (train)
        for other_agent in agents:
            src_lbl_train = os.path.join(base_source_dir, other_agent, 'train', 'labels')
            tgt_lbl_train = os.path.join(base_target_dir, f"{agent}_synthesized", 'train', 'labels')
            copy_images_and_labels(src_lbl_train, tgt_lbl_train)
    
    # Validation data
    for agent in agents:
        src_images_val = os.path.join(base_source_dir, agent, 'valid', 'images')
        tgt_images_val = os.path.join(base_target_dir, f"{agent}_synthesized", 'valid', 'images')
        synthesize_images(model, src_images_val, tgt_images_val)
        
        # Copy images+labels from other agents (valid)
        for other_agent in agents:
            if other_agent != agent:
                other_src_val = os.path.join(base_source_dir, other_agent, 'valid', 'images')
                copy_images_and_labels(other_src_val, tgt_images_val)
        
        # Copy labels from all agents (valid)
        for other_agent in agents:
            src_lbl_val = os.path.join(base_source_dir, other_agent, 'valid', 'labels')
            tgt_lbl_val = os.path.join(base_target_dir, f"{agent}_synthesized", 'valid', 'labels')
            copy_images_and_labels(src_lbl_val, tgt_lbl_val)

    print("Dataset generation complete!")

if __name__ == "__main__":
    generate_synthesized_dataset()

In [ ]:
import cv2

# Load the image
test_image_path = "../data/synthetic_raw/encoder-decoder/agent_0_synthesized/train/images/train_b_cell_11_4_23_cap1_1011.png"
image = cv2.imread(test_image_path)

# Check if the image was loaded successfully
if image is None:
    print("Error: Unable to load image.")
else:
    # Get the number of channels
    num_channels = image.shape[2] if len(image.shape) == 3 else 1
    print(f"The image has {num_channels} channels.")

    # Check if all channels have the same values for each pixel
    if num_channels > 1:
        # Compare all channels
        is_identical = (image[:, :, 0] == image[:, :, 1]).all() and (image[:, :, 1] == image[:, :, 2]).all()
        if is_identical:
            print("All channels have the same values for each pixel.")
        else:
            print("The channels do not have the same values for each pixel.")
    else:
        print("The image is grayscale (only one channel).")